In [ ]:
# -*- coding: utf-8 -*-
import gc
import hashlib
import json
import os
import random
import shutil
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.datasets as D
import torchvision.models as M
import torchvision.transforms as T
from PIL import ImageEnhance, ImageFilter
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)
from torch.utils.data import DataLoader, Subset


# ============================================================================
# 全域參數與超參數設定
# ============================================================================
SEED = 42
DATA = "./data"
OUT = Path("outputs")

NC = 43
BS = 64
EVAL_BS = 128
EPOCHS = 15
LR = 5e-4
WD = 1e-4


NW = 4
PIN_MEMORY = True
PERSISTENT_WORKERS = (NW > 0)

EXPECTED_TEST = 12630
KNOWN_TRAIN_SIZES = {26640}

CLASS_NAMES = [
    "Speed limit 20", "Speed limit 30", "Speed limit 50", "Speed limit 60",
    "Speed limit 70", "Speed limit 80", "End speed limit 80", "Speed limit 100",
    "Speed limit 120", "No passing", "No passing >3.5t",
    "Right-of-way next intersection", "Priority road", "Yield", "Stop",
    "No vehicles", "Vehicles >3.5t prohibited", "No entry", "General caution",
    "Dangerous curve left", "Dangerous curve right", "Double curve", "Bumpy road",
    "Slippery road", "Road narrows right", "Road work", "Traffic signals",
    "Pedestrians", "Children crossing", "Bicycles crossing", "Beware ice/snow",
    "Wild animals crossing", "End all limits", "Turn right ahead",
    "Turn left ahead", "Ahead only", "Go straight or right",
    "Go straight or left", "Keep right", "Keep left", "Roundabout mandatory",
    "End no passing", "End no passing >3.5t",
]

FORCE_RETRAIN = False

CONFIGS = [
    {"name": "baseline_cnn_64", "arch": "baseline_cnn", "size": 64, "cw": False},
    {"name": "resnet18_scratch_64", "arch": "resnet18_scratch", "size": 64, "cw": False},
    {"name": "resnet18_64", "arch": "resnet18", "size": 64, "cw": False},
    {"name": "resnet18_128", "arch": "resnet18", "size": 128, "cw": False},
    {"name": "efficientnet_b0_128", "arch": "efficientnet_b0", "size": 128, "cw": False},
]
ROBUST_MODELS = ["baseline_cnn_64", "resnet18_scratch_64", "resnet18_64"]

DEGRADATIONS = {
    "brightness": [0.3, 0.5, 0.7],
    "blur": [1.0, 2.0, 4.0],
    "rotation": [5, 10, 15],
}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CFG = {c["name"]: c for c in CONFIGS}

TR_IDX, VA_IDX = None, None
SPLIT_SIG = None
_LOADERS = {}


def setup_runtime():
    OUT.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True


def check_env():
    print("=" * 60)
    print(f"Python      : {sys.version.split()[0]}")
    print(f"PyTorch     : {torch.__version__} | torchvision: {torchvision_ver()}")
    print(f"OS          : {os.name} ({sys.platform}) | DataLoader workers: {NW}")
    if torch.cuda.is_available():
        prop = torch.cuda.get_device_properties(0)
        print(f"GPU         : {torch.cuda.get_device_name(0)} ({prop.total_memory / 1e9:.2f} GB)")
        print(f"CUDA        : {torch.version.cuda} | TF32 Enabled: {torch.backends.cuda.matmul.allow_tf32}")
    else:
        print("未偵測到CUDA GPU，會使用CPU執行")
    print("=" * 60)


def torchvision_ver():
    import torchvision
    return torchvision.__version__


# ============================================================================
# 分層切分
# ============================================================================
def eval_pipeline(size):
    return [
        T.Resize((size, size)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]


def train_pipeline(size):
    return T.Compose([
        T.Resize((size, size)),
        T.RandomRotation(15),
        T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        T.RandomErasing(p=0.2, scale=(0.02, 0.2), value="random"),
    ])


GTSRB_BASE = "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/"
GTSRB_ZIPS = {
    "GTSRB-Training_fixed.zip": "GTSRB/Training",
    "GTSRB_Final_Test_Images.zip": "GTSRB/Final_Test/Images",
    "GTSRB_Final_Test_GT.zip": "GT-final_test.csv",
}


def _download(url, dst, timeout=180, tries=5):
    for i in range(1, tries + 1):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as r, open(dst, "wb") as f:
                shutil.copyfileobj(r, f, 1 << 20)
            return
        except Exception as e:
            print(f"    第 {i}/{tries} 次下載失敗：{type(e).__name__}: {e}")
            if i < tries:
                time.sleep(5 * i)
    raise RuntimeError(
        f"下載 {url} 失敗。"
        f"請手動將以下壓縮檔放進本地 {Path(DATA) / 'gtsrb'} 後重跑："
        f"  1. GTSRB-Training_fixed.zip"
        f"  2. GTSRB_Final_Test_Images.zip"
        f"  3. GTSRB_Final_Test_GT.zip"
        )


def ensure_data():
    base = Path(DATA) / "gtsrb"
    need = {base / rel: z for z, rel in GTSRB_ZIPS.items()}
    if all(p.exists() for p in need):
        return base

    base.mkdir(parents=True, exist_ok=True)
    for target, zname in need.items():
        if target.exists():
            continue
        zp = base / zname
        if not zp.exists():
            print(f"[data] 未找到 {zname}，開始從官方伺服器下載...")
            _download(GTSRB_BASE + zname, zp)
            print(f"[data] 下載完成 {zname} ({zp.stat().st_size / 1e6:.1f} MB)")
        print(f"[data] 正在解壓縮 {zname} ...")
        try:
            with zipfile.ZipFile(zp) as zf:
                zf.extractall(base)
        except zipfile.BadZipFile:
            zp.unlink()
            raise RuntimeError(f"{zname} 壓縮檔損毀，已自動刪除，請重新執行。")

    missing = [str(p) for p in need if not p.exists()]
    if missing:
        raise RuntimeError("解壓後缺少必要目錄或檔案：  " + "  ".join(missing))
    return base


def check_dataset():
    ensure_data()
    raw = D.GTSRB(root=DATA, split="train", download=False)
    D.GTSRB(root=DATA, split="test", download=False)
    n = len(raw)
    if n not in KNOWN_TRAIN_SIZES:
        raise RuntimeError(
            f"GTSRB train 有 {n} 張，不是 torchvision fixed 版的 26,640。"
            f"多半是 data/ 解壓中斷；請刪掉 data/gtsrb 資料夾後重跑。"
        )
    print(f"[data] 資料集驗證通過：torchvision fixed 版，共 {n} 張訓練影像（43 類）")
    return raw


def build_split():
    global SPLIT_SIG
    raw = check_dataset()
    samples = getattr(raw, "_samples", None) or raw.samples
    paths = np.array([Path(s[0]).as_posix() for s in samples])
    labels = np.array([s[1] for s in samples])
    groups = np.array([
        f"{Path(p).parent.name}/{Path(p).stem.split('_')[0]}" for p in paths
    ])

    rng = np.random.default_rng(SEED)
    tr_parts, va_parts = [], []
    for c in range(NC):
        idx_c = np.where(labels == c)[0]
        tracks = sorted({groups[i] for i in idx_c})
        rng.shuffle(tracks)
        n_val = max(1, round(0.2 * len(tracks)))
        va_tracks = set(tracks[:n_val])
        is_va = np.array([groups[i] in va_tracks for i in idx_c])
        va_parts.append(idx_c[is_va])
        tr_parts.append(idx_c[~is_va])
    tr_idx, va_idx = np.sort(np.concatenate(tr_parts)), np.sort(np.concatenate(va_parts))

    per_class_va = np.bincount(labels[va_idx], minlength=NC)
    assert per_class_va.min() >= 1, "val 有類別 0 張"
    assert len(np.unique(labels[tr_idx])) == NC, "train 有類別 0 張"
    assert not (set(groups[tr_idx]) & set(groups[va_idx])), "同一支影片同時出現在 train 與 val"
    print(f"[split] train: {len(tr_idx)} | val: {len(va_idx)} | tracks: {len(set(groups))} | "
          f"每類 val {per_class_va.min()}–{per_class_va.max()} 張")

    np.save(OUT / "split_train_idx.npy", tr_idx)
    np.save(OUT / "split_val_idx.npy", va_idx)
    pd.DataFrame({"index": range(NC), "sign": CLASS_NAMES}).to_csv(
        OUT / "class_names.csv", index=False)

    SPLIT_SIG = hashlib.md5(np.ascontiguousarray(tr_idx).tobytes()).hexdigest()[:8]
    print(f"[split] 切分簽章 SPLIT_SIG = {SPLIT_SIG}")
    return tr_idx, va_idx


def init_data():
    global TR_IDX, VA_IDX
    if TR_IDX is None:
        TR_IDX, VA_IDX = build_split()
    return TR_IDX, VA_IDX


def get_loaders(size):
    init_data()
    if size in _LOADERS:
        return _LOADERS[size]
    tr_raw = D.GTSRB(root=DATA, split="train", download=False, transform=train_pipeline(size))
    va_raw = D.GTSRB(root=DATA, split="train", download=False, transform=T.Compose(eval_pipeline(size)))
    te_raw = D.GTSRB(root=DATA, split="test", download=False, transform=T.Compose(eval_pipeline(size)))
    if len(te_raw) != EXPECTED_TEST:
        raise RuntimeError(f"GTSRB test 有 {len(te_raw)} 張，應為 {EXPECTED_TEST} 張。")

    loader_kwargs = {
        "num_workers": NW,
        "pin_memory": PIN_MEMORY and torch.cuda.is_available(),
        "persistent_workers": PERSISTENT_WORKERS,
    }
    _LOADERS[size] = (
        DataLoader(Subset(tr_raw, TR_IDX), batch_size=BS, shuffle=True, drop_last=True, **loader_kwargs),
        DataLoader(Subset(va_raw, VA_IDX), batch_size=EVAL_BS, shuffle=False, **loader_kwargs),
        DataLoader(te_raw, batch_size=EVAL_BS, shuffle=False, **loader_kwargs),
    )
    return _LOADERS[size]


# ============================================================================
# 模型定義與訓練流程
# ============================================================================
class BaselineCNN(nn.Module):
    """3 層卷積 + 4×4 adaptive average pool + FC（約 0.63M 參數、from scratch）"""

    def __init__(self, num_classes=NC):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128 * 4 * 4, 256), nn.ReLU(),
            nn.Dropout(0.4), nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def build_model(arch):
    if arch == "baseline_cnn":
        return BaselineCNN()
    if arch == "resnet18_scratch":
        m = M.resnet18(weights=None)
        m.fc = nn.Linear(m.fc.in_features, NC)
        return m
    if arch == "resnet18":
        m = M.resnet18(weights=M.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, NC)
        return m
    if arch == "efficientnet_b0":
        m = M.efficientnet_b0(weights=M.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, NC)
        return m
    raise ValueError(f"unknown arch: {arch}")


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def read_history(name):
    p = OUT / f"history_{name}.json"
    if not p.exists():
        return {"best": {"epoch": 0, "val_acc": 0.0}, "minutes": 0.0, "params_m": 0.0}
    with open(p, encoding="utf-8") as f:
        return json.load(f)


def train_one(cfg):
    name, size = cfg["name"], cfg["size"]
    init_data()
    ckpt = OUT / f"{name}.pt"
    if ckpt.exists() and not FORCE_RETRAIN:
        old = read_history(name).get("split_sig")
        if old == SPLIT_SIG:
            print(f"[skip] {name} 已有對應 split ({SPLIT_SIG}) 的 checkpoint，跳過訓練")
            return
        print(f"[retrain] {name} 的舊 checkpoint (split={old}) 與目前 ({SPLIT_SIG}) 不符，重新訓練")

    tr_loader, va_loader, _ = get_loaders(size)
    model = build_model(cfg["arch"]).to(DEV)
    print(f"=== 開始訓練 {name} | 架構: {cfg['arch']} @{size}x{size} | "
          f"參數量: {count_params(model)/1e6:.2f}M ===")

    crit = nn.CrossEntropyLoss()
    if cfg.get("cw"):
        _ds = tr_loader.dataset
        _samples = getattr(_ds.dataset, "_samples", None) or _ds.dataset.samples
        _cnt = np.bincount([_samples[i][1] for i in _ds.indices], minlength=NC)
        _w = len(_ds.indices) / (NC * np.maximum(_cnt, 1))
        crit = nn.CrossEntropyLoss(weight=torch.tensor(_w, dtype=torch.float32, device=DEV))

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

    hist = {k: [] for k in ("train_loss", "val_loss", "train_acc", "val_acc")}
    hist["best"] = {"epoch": 0, "val_acc": 0.0}
    hist["split_sig"] = SPLIT_SIG
    t0 = time.time()

    for ep in range(1, EPOCHS + 1):
        model.train()
        tl = tc = tt = 0
        for x, y in tr_loader:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            out = model(x)
            loss = crit(out, y)
            loss.backward()
            opt.step()
            tl += loss.item() * y.size(0)
            tc += (out.argmax(1) == y).sum().item()
            tt += y.size(0)

        model.eval()
        vl = vc = vt = 0
        with torch.no_grad():
            for x, y in va_loader:
                x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
                out = model(x)
                vl += crit(out, y).item() * y.size(0)
                vc += (out.argmax(1) == y).sum().item()
                vt += y.size(0)
        sch.step()

        hist["train_loss"].append(tl / tt)
        hist["train_acc"].append(tc / tt)
        hist["val_loss"].append(vl / vt)
        hist["val_acc"].append(vc / vt)

        star = ""
        if vc / vt > hist["best"]["val_acc"]:
            hist["best"] = {"epoch": ep, "val_acc": vc / vt}
            torch.save(model.state_dict(), ckpt)
            star = "  * [已存最佳權重]"
        print(f"ep {ep:02d}/{EPOCHS} | train {tl/tt:.4f} ({tc/tt*100:5.2f}%) | "
              f"val {vl/vt:.4f} ({vc/vt*100:5.2f}%) | {time.time()-t0:5.1f}s{star}")

    hist["minutes"] = (time.time() - t0) / 60
    hist["params_m"] = count_params(model) / 1e6
    with open(OUT / f"history_{name}.json", "w", encoding="utf-8") as f:
        json.dump(hist, f, indent=1)
    print(f"--> {name} 訓練完成：最佳 val {hist['best']['val_acc']*100:.2f}% "
          f"(Epoch {hist['best']['epoch']})，耗時 {hist['minutes']:.2f} 分鐘")

    del model, opt, crit
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================================
# 評估、統計檢定與圖表繪製
# ============================================================================
def load_ckpt(cfg):
    p = OUT / f"{cfg['name']}.pt"
    if not p.exists():
        raise FileNotFoundError(f"缺少 {p}，請先執行訓練。")
    model = build_model(cfg["arch"]).to(DEV)
    try:
        state = torch.load(p, map_location=DEV, weights_only=True)
    except TypeError:
        state = torch.load(p, map_location=DEV)
    model.load_state_dict(state)
    model.eval()
    return model


@torch.no_grad()
def predict(model, loader):
    labels, probs = [], []
    for x, y in loader:
        probs.append(model(x.to(DEV, non_blocking=True)).softmax(1).cpu())
        labels.append(y)
    return torch.cat(labels).numpy(), torch.cat(probs).numpy()


def plot_curves():
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for cfg in CONFIGS:
        p = OUT / f"history_{cfg['name']}.json"
        if not p.exists():
            continue
        h = read_history(cfg["name"])
        if "train_acc" not in h:
            continue
        ep = range(1, len(h["train_acc"]) + 1)
        axes[0].plot(ep, h["train_loss"], "--", alpha=.6, label=f"{cfg['name']} train")
        axes[0].plot(ep, h["val_loss"], "-o", ms=3, label=f"{cfg['name']} val")
        axes[1].plot(ep, np.array(h["train_acc"]) * 100, "--", alpha=.6, label=f"{cfg['name']} train")
        axes[1].plot(ep, np.array(h["val_acc"]) * 100, "-o", ms=3, label=f"{cfg['name']} val")
        axes[1].scatter(h["best"]["epoch"], h["best"]["val_acc"] * 100, marker="*", s=160, zorder=5)
    axes[0].set(xlabel="epoch", ylabel="cross-entropy loss", title="Loss")
    axes[1].set(xlabel="epoch", ylabel="accuracy (%)", title="Accuracy (star = selected epoch)")
    for ax in axes:
        ax.grid(alpha=.3)
        ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(OUT / "fig_curves.png", dpi=150)
    plt.close(fig)


def plot_confusion(labels, pred, name):
    cm = confusion_matrix(labels, pred, labels=range(NC))
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    im = ax.imshow(np.log1p(cm), cmap="viridis")
    fig.colorbar(im, ax=ax, label="log(1 + count)")
    ax.set(xlabel="predicted class", ylabel="true class", title=f"Confusion matrix — {name}")
    fig.tight_layout()
    fig.savefig(OUT / f"fig_confusion_{name}.png", dpi=150)
    plt.close(fig)

    off = cm.copy()
    np.fill_diagonal(off, 0)
    pairs = np.dstack(np.unravel_index(np.argsort(off.ravel())[::-1][:10], off.shape))[0]
    rows = [{"true": int(i), "true_sign": CLASS_NAMES[i],
             "pred": int(j), "pred_sign": CLASS_NAMES[j], "count": int(off[i, j])}
            for i, j in pairs]
    pd.DataFrame(rows).to_csv(OUT / f"confused_pairs_{name}.csv", index=False)
    return rows


def bootstrap_ci(labels, pred, n=2000):
    rng = np.random.default_rng(SEED)
    ok = pred == labels
    accs = [ok[rng.integers(0, len(ok), len(ok))].mean() * 100 for _ in range(n)]
    return float(np.percentile(accs, 2.5)), float(np.percentile(accs, 97.5))


def mcnemar(a_pred, b_pred, labels):
    from scipy.stats import binomtest

    a_ok, b_ok = a_pred == labels, b_pred == labels
    a_only, b_only = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    n = a_only + b_only
    return a_only, b_only, (binomtest(min(a_only, b_only), n, 0.5).pvalue if n else 1.0)


def plot_coverage(name, val_labels, val_probs, te_labels, te_probs, risk_target=0.01):
    v_conf, v_ok = val_probs.max(1), val_probs.argmax(1) == val_labels
    t_conf, t_ok = te_probs.max(1), te_probs.argmax(1) == te_labels

    tau = 0.999
    for t in np.linspace(0.3, 0.999, 400):
        m = v_conf >= t
        if m.sum() >= 100 and 1 - v_ok[m].mean() <= risk_target:
            tau = float(t)
            break

    m = t_conf >= tau
    cov = m.mean() * 100
    acc = t_ok[m].mean() * 100 if m.any() else float("nan")

    order = np.argsort(-t_conf)
    risk = 1 - np.cumsum(t_ok[order]) / np.arange(1, len(t_ok) + 1)
    aurc = float(risk.mean() * 100)

    t90 = float(np.quantile(t_conf, 0.10))
    acc90 = t_ok[t_conf >= t90].mean() * 100

    fig, ax = plt.subplots(figsize=(5.6, 4.2))
    ax.plot(np.linspace(0, 100, len(risk)), (1 - risk) * 100, lw=2, label="test")
    ax.scatter([cov], [acc], marker="*", s=200, zorder=5,
               label=f"τ={tau:.3f} (val-selected): {cov:.1f}% / {acc:.2f}%")
    ax.set(xlabel="coverage (% of test images answered)",
           ylabel="accuracy on answered (%)",
           title=f"{name} — AURC {aurc:.2f}")
    ax.grid(alpha=.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / f"fig_coverage_{name}.png", dpi=150)
    plt.close(fig)

    return {"model": name, "tau_val": round(tau, 3), "coverage_%": round(cov, 2),
            "accuracy_%": round(acc, 2), "selective_risk_%": round(100 - acc, 2),
            "aurc_%": round(aurc, 2), "acc_at_90cov_oracle_%": round(acc90, 2)}


def plot_samples(names=("resnet18_scratch_64", "resnet18_64"), k=3):
    names = tuple(n for n in names if (OUT / f"{n}.pt").exists())
    if not names:
        return
    models = {n: load_ckpt(CFG[n]) for n in names}
    ds = {CFG[n]["size"]: D.GTSRB(root=DATA, split="test", download=False,
                                  transform=T.Compose(eval_pipeline(CFG[n]["size"])))
          for n in names}
    rng = random.Random(SEED)
    idxs = rng.sample(range(len(next(iter(ds.values())))), k)
    fig, axes = plt.subplots(1, k, figsize=(4.2 * k, 4.6))
    for ax, i in zip(np.atleast_1d(axes), idxs):
        lines, shown = [], None
        for n, m in models.items():
            t, label = ds[CFG[n]["size"]][i]
            shown = t if shown is None else shown
            with torch.no_grad():
                p = m(t.unsqueeze(0).to(DEV)).softmax(1)
            c, k_ = p.max(1)
            lines.append(f"{n}: {CLASS_NAMES[k_.item()]} ({c.item()*100:.1f}%)")
        img = shown.permute(1, 2, 0).numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        ax.imshow(np.clip(img, 0, 1))
        ax.set_title(f"truth: {CLASS_NAMES[label]}\n" + "\n".join(lines), fontsize=7)
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(OUT / "fig_samples.png", dpi=150)
    plt.close(fig)
    for m in models.values():
        del m


def run_test_phase():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    init_data()

    rows, payload, cov_rows = [], {}, []
    for cfg in CONFIGS:
        if not (OUT / f"{cfg['name']}.pt").exists():
            print(f"[skip] {cfg['name']} 沒有 checkpoint")
            continue
        model = load_ckpt(cfg)
        _, va_loader, te_loader = get_loaders(cfg["size"])
        va_labels, va_probs = predict(model, va_loader)
        labels, probs = predict(model, te_loader)
        pred = probs.argmax(1)

        acc = accuracy_score(labels, pred) * 100
        lo, hi = bootstrap_ci(labels, pred)
        p, r, f, _ = precision_recall_fscore_support(labels, pred, average="macro", zero_division=0)
        h = read_history(cfg["name"])
        if h.get("split_sig") != SPLIT_SIG:
            print(f"[warn] {cfg['name']} 的 checkpoint split ({h.get('split_sig')}) 與目前 ({SPLIT_SIG}) 不同！")
        rows.append({
            "model": cfg["name"], "size": cfg["size"], "params_M": round(h.get("params_m", 0), 2),
            "train_min": round(h.get("minutes", 0), 1),
            "best_val_acc": round(h["best"]["val_acc"] * 100, 2),
            "test_acc": round(acc, 2), "acc_ci95_low": round(lo, 2), "acc_ci95_high": round(hi, 2),
            "macro_precision": round(p * 100, 2),
            "macro_recall": round(r * 100, 2), "macro_f1": round(f * 100, 2),
            "val_test_gap": round(h["best"]["val_acc"] * 100 - acc, 2),
        })
        payload[cfg["name"]] = (labels, probs)
        np.save(OUT / f"pred_{cfg['name']}.npy", pred)

        plot_confusion(labels, pred, cfg["name"])
        cov = plot_coverage(cfg["name"], va_labels, va_probs, labels, probs)
        cov_rows.append(cov)
        print(f"[{cfg['name']}] Test Acc: {acc:.2f}% (95% CI: {lo:.2f}–{hi:.2f}%)")
        print(f"  τ={cov['tau_val']} (val 選出) -> Test Coverage: {cov['coverage_%']}% / Acc: {cov['accuracy_%']}% | AURC: {cov['aurc_%']}")

        per_class = precision_recall_fscore_support(labels, pred, labels=range(NC), zero_division=0)
        pc = pd.DataFrame({"class": range(NC), "sign": CLASS_NAMES,
                           "precision": per_class[0], "recall": per_class[1],
                           "f1": per_class[2], "support": per_class[3]})
        pc.to_csv(OUT / f"per_class_{cfg['name']}.csv", index=False)
        worst = pc.nsmallest(5, "recall")
        print("  最差 5 類 recall: " + ", ".join(
            f"{row.sign} {row.recall*100:.1f}% (n={row.support})" for row in worst.itertuples()))

        del model, labels, pred, va_labels, va_probs
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if not rows:
        print("沒有任何checkpoint，請先訓練。")
        return pd.DataFrame(), {}

    df = pd.DataFrame(rows)
    df.to_csv(OUT / "results.csv", index=False)
    pd.DataFrame(cov_rows).to_csv(OUT / "coverage.csv", index=False)
    print("=== RESULTS (Test Set 評估總表) ===")
    try:
        print(df.to_markdown(index=False))
    except Exception:
        print(df.to_string(index=False))

    names = list(payload)
    sig = []
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            la, pa = payload[a]
            a_only, b_only, pv = mcnemar(pa.argmax(1), payload[b][1].argmax(1), la)
            sig.append({"model_a": a, "model_b": b, "a_only_correct": a_only,
                        "b_only_correct": b_only, "p_mcnemar": round(pv, 4)})
    if sig:
        print("=== 配對顯著性檢定（McNemar, test n=12,630）===")
        print(pd.DataFrame(sig).to_string(index=False))
        pd.DataFrame(sig).to_csv(OUT / "significance.csv", index=False)

    plot_curves()
    plot_samples()
    return df, payload


# ============================================================================
# 退化穩健性測試 (Robustness Phase)
# ============================================================================
class _Brightness:

    def __init__(self, factor):
        self.factor = factor

    def __call__(self, im):
        return ImageEnhance.Brightness(im).enhance(self.factor)


class _Blur:
    def __init__(self, radius):
        self.radius = radius

    def __call__(self, im):
        return im.filter(ImageFilter.GaussianBlur(radius=self.radius))


def degradation(kind, value):
    if kind == "brightness":
        return _Brightness(value)
    if kind == "blur":
        return _Blur(value)
    if kind == "rotation":
        return T.RandomRotation((value, value))
    raise ValueError(kind)


@torch.no_grad()
def eval_degraded(model, cfg, kind, value):
    tf = T.Compose([degradation(kind, value)] + eval_pipeline(cfg["size"]))
    ds = D.GTSRB(root=DATA, split="test", download=False, transform=tf)
    dl = DataLoader(
        ds, batch_size=EVAL_BS, shuffle=False,
        num_workers=NW, pin_memory=PIN_MEMORY and torch.cuda.is_available()
    )
    correct = total = 0
    for x, y in dl:
        correct += (model(x.to(DEV, non_blocking=True)).argmax(1).cpu() == y).sum().item()
        total += y.size(0)
    return correct / total * 100


def run_robustness_phase(results_df):
    if results_df.empty:
        return
    print("=== 開始退化穩健性測試 (Robustness Phase) ===")
    acc = dict(zip(results_df["model"], results_df["test_acc"]))
    rows = []
    for name in ROBUST_MODELS:
        if name not in CFG or name not in acc:
            continue
        cfg = CFG[name]
        model = load_ckpt(cfg)
        rows.append({"model": name, "degradation": "none", "level": 0,
                     "value": 0, "acc": acc[name]})
        for kind, levels in DEGRADATIONS.items():
            for lv, v in enumerate(levels, 1):
                a = eval_degraded(model, cfg, kind, v)
                rows.append({"model": name, "degradation": kind, "level": lv,
                             "value": v, "acc": round(a, 2)})
                print(f"  {name:<20} | {kind:<10} lv.{lv} (val={v}) -> {a:.2f}%")
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    df.to_csv(OUT / "robustness.csv", index=False)

    used = [n for n in ROBUST_MODELS if n in acc] or [df["model"].iloc[0]]
    fig, axes = plt.subplots(1, len(used), figsize=(5.4 * len(used), 4), squeeze=False)
    for ax, name in zip(axes[0], used):
        sub = df[df["model"] == name]
        for kind, g in sub[sub["level"] > 0].groupby("degradation"):
            ax.plot(g["level"], g["acc"], "-o", label=kind)
        ax.axhline(acc[name], ls="--", c="gray", label="clean")
        ax.set(xlabel="severity (1=light, 3=heavy)", ylabel="test accuracy (%)",
               title=name, xticks=[1, 2, 3])
        ax.grid(alpha=.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "fig_robustness.png", dpi=150)
    plt.close(fig)
    print("=== 穩健性測試彙整表 ===")
    print(df.pivot_table(index="model", columns=["degradation", "level"], values="acc").to_string())


# ============================================================================
# 環境紀錄、煙霧測試與主程式入口
# ============================================================================
def dump_env():
    init_data()
    info = {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "torchvision": torchvision_ver(),
        "device": str(DEV),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "cuda": torch.version.cuda,
        "tf32_enabled": bool(torch.cuda.is_available() and torch.backends.cuda.matmul.allow_tf32),
        "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),
        "gtsrb_train": len(TR_IDX) + len(VA_IDX),
        "gtsrb_test": EXPECTED_TEST,
        "split": {"train": len(TR_IDX), "val": len(VA_IDX), "sig": SPLIT_SIG},
        "epochs": EPOCHS,
        "batch": BS,
        "eval_batch": EVAL_BS,
        "lr": LR,
        "weight_decay": WD,
        "optimizer": "AdamW",
        "scheduler": "CosineAnnealingLR",
    }
    (OUT / "env.txt").write_text(json.dumps(info, indent=2), encoding="utf-8")
    print("[env] 環境資訊已寫入 outputs/env.txt")


def selftest():
    """煙霧測試：不使用GTSRB資料集，確認模型、VRAM 與繪圖流程正常"""
    print("[selftest] 用隨機張量檢查模型與評估流程...")
    for cfg in CONFIGS:
        m = build_model(cfg["arch"]).to(DEV)
        x = torch.randn(8, 3, cfg["size"], cfg["size"], device=DEV)
        y = torch.randint(0, NC, (8,), device=DEV)
        loss = nn.CrossEntropyLoss()(m(x), y)
        loss.backward()
        with torch.no_grad():
            p = m(x[:2]).softmax(1).cpu().numpy()
        assert p.shape == (2, NC), p.shape
        assert np.allclose(p.sum(1), 1, atol=1e-4), p.sum(1)
        print(f"  {cfg['name']:<22} forward/backward OK | "
              f"{count_params(m)/1e6:.2f}M params | loss {loss.item():.3f}")
        del m

    labels = np.array([0, 1, 1])
    probs = np.array([[.9, .05, .05], [.2, .7, .1], [.1, .1, .8]])
    plot_confusion(labels, probs.argmax(1), "selftest")

    rng = np.random.default_rng(SEED)
    lab = rng.integers(0, NC, 300)
    logits = rng.normal(0, 1, (300, NC))
    logits[np.arange(300), lab] += 6
    pr = np.exp(logits) / np.exp(logits).sum(1, keepdims=True)
    cov = plot_coverage("selftest", lab, pr, lab, pr)
    assert 0 <= cov["coverage_%"] <= 100 and 0 <= cov["accuracy_%"] <= 100, cov

    for f in ("fig_coverage_selftest.png", "fig_confusion_selftest.png",
              "confused_pairs_selftest.csv"):
        (OUT / f).unlink(missing_ok=True)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("[selftest] 環境與預訓練權重正常。")


def main():
    setup_runtime()
    check_env()

    if len(sys.argv) > 1 and sys.argv[1] == "selftest":
        selftest()
        return

    selftest()

    dump_env()

    for cfg in CONFIGS:
        train_one(cfg)

    df, _ = run_test_phase()

    run_robustness_phase(df)

    shutil.make_archive("outputs", "zip", root_dir=".", base_dir="outputs")

    print("" + "=" * 60)
    print("實驗完成！檔案位於 outputs/ 資料夾（並已打包為 outputs.zip）：")
    for f in sorted(OUT.iterdir()):
        print(f"   - {f.name}")
    print("=" * 60)


if __name__ == "__main__":
    main()